# 00 — Unified pipeline (V-JEPA + DINOv2)

Single run: **data check → shared YOLO crops → dual embeddings → dual train → analysis → comparison**.

| Step | What |
|------|------|
| 01 | InHARD inventory |
| 02 | Embeddings: `embeddings.npz` + `embeddings_dinov2.npz` (crops extracted once) |
| 03 | Train: `har_vjepa_*` + `har_dinov2_*` |
| 06 | Extended analysis charts per backbone |
| 07 | Side-by-side comparison + winner |

Edit **cell 2** (`CLIPS_PER_CLASS`, default **100**). Other notebooks (02–07) remain for standalone steps; **08** is optional (same as this run).

In [1]:
import sys
from pathlib import Path

NB = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
sys.path.insert(0, str(NB))
from lib.paths import ensure_notebook_paths
ensure_notebook_paths()
from lib.reload import reload_lib_modules
reload_lib_modules()

from lib.constants import BACKBONE_DINOV2, BACKBONE_VJEPA, BLOCKED_ACTIONS, TRAINABLE_ACTIONS
from lib.inhard import analyze_training_clips, label_counts, subject_counts
from lib.pipeline import PipelineConfig, run_tag, iter_backbone_cfgs
from lib.paths import find_inhard_root, CHECKPOINTS_DIR, OUTPUTS_DIR

In [2]:
# ── Configuration ────────────────────────────────────────────────────────────
CLIPS_PER_CLASS = 100     # 1 = smoke test · 5 = quick · 100 = paper run
SAMPLE_SEED = 42
TRAIN_EPOCHS = 25
ANALYSIS_SPLIT = "random"   # holdout for analysis + comparison
# ─────────────────────────────────────────────────────────────────────────────

RUN_TAG = run_tag(PipelineConfig(clips_per_class=CLIPS_PER_CLASS))

CFG = PipelineConfig(
    clips_per_class=CLIPS_PER_CLASS,
    sample_seed=SAMPLE_SEED,
    train_epochs=TRAIN_EPOCHS,
    backbones=(BACKBONE_VJEPA, BACKBONE_DINOV2),
    exclude_train=BLOCKED_ACTIONS,
    exclude_infer=BLOCKED_ACTIONS,
    min_train_classes=len(TRAINABLE_ACTIONS),
    embedding_mode="yolo_crop",
    split_mode="subject",
    analysis_split=ANALYSIS_SPLIT,
    run_analysis=True,
    run_compare=True,
    run_eval_video=False,
    skip_embeddings_if_exists=False,
    skip_train_if_checkpoint_exists=False,
)

print("InHARD:", find_inhard_root())
print(f"Run tag: {RUN_TAG}")
print("Checkpoints:")
for c in iter_backbone_cfgs(CFG):
    print(f"  {c.backbone}: {c.checkpoint_name}")

_prev = analyze_training_clips(
    exclude_labels=CFG.exclude_train,
    clips_per_class=CFG.clips_per_class,
    min_classes=CFG.min_train_classes,
    seed=CFG.sample_seed,
)
print(f"Preview: {_prev.n_classes} classes, {len(_prev.clips)} clips")
if _prev.ok:
    print(subject_counts(_prev.clips).head(8).to_string())

InHARD: /Volumes/Carlos Pano HD/MASTER-AI/PROYECTO INTEGRADOR/IN-HARD/01-InHARD
Run tag: 12c_crop_100each
Checkpoints:
  vjepa: har_vjepa_12c_crop_100each.pt
  dinov2: har_dinov2_12c_crop_100each.pt
Preview: 12 classes, 1066 clips
P05    129
P08    124
P03     91
P10     77
P01     71
P09     66
P07     61
P14     61


## Run full pipeline

In [3]:
from lib.pipeline import run_pipeline

results = run_pipeline(CFG)
results

[01] Sample: 1066 clips, 12 classes
[01]   100 clips/class · mode=yolo_crop
[01]   Labels: Consult sheets, Picking in front, Picking left, Put down component, Put down measuring rod, Put down screwdriver…
[01] Mock videos: ['Industrial-One.mp4', 'madera.mp4']
[02] Shared crop pass · 1066 clips · backbones=['vjepa', 'dinov2']


YOLO crops:   0%|          | 0/1066 [00:00<?, ?it/s]

[02] Encoding 1066 clips · backbone=vjepa


vjepa encode:   0%|          | 0/1066 [00:00<?, ?it/s]

[02] Merged 5 human-verified samples (vjepa)
[02] Saved (1070, 1024) → embeddings.npz (+5 human)
[02] Encoding 1066 clips · backbone=dinov2


dinov2 encode:   0%|          | 0/1066 [00:00<?, ?it/s]

Using cache found in /Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/cpanoh/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


[02] Merged 5 human-verified samples (dinov2)
[02] Saved (1071, 1024) → embeddings_dinov2.npz (+5 human)
[03] ── backbone=vjepa ──
epoch 5/25  train=2.1302  val=2.2978
epoch 10/25  train=1.8939  val=2.3299
epoch 15/25  train=1.7565  val=2.2097
epoch 20/25  train=1.6492  val=2.1399
epoch 25/25  train=1.5826  val=2.0370
[03] Checkpoint → /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/checkpoints/har_vjepa_12c_crop_100each.pt (split=subject, weights=True)
[03] ── backbone=dinov2 ──
epoch 5/25  train=1.8427  val=2.0450
epoch 10/25  train=1.4909  val=2.0740
epoch 15/25  train=1.2755  val=1.8439
epoch 20/25  train=1.0818  val=1.7982
epoch 25/25  train=0.9624  val=2.0958
[03] Checkpoint → /Users/cpanoh/Documents/cpano-98-local/GitHub/AI-Co-Pilot-for-the-Production-Floor-See-Guide-Improve/notebooks/checkpoints/har_dinov2_12c_crop_100each.pt (split=subject, weights=True)
[06] Analysis vjepa · split=random
[06] Analysis dinov2 · spl

{'config': {'exclude_train': ('Assemble system', 'No action'),
  'exclude_infer': ('Assemble system', 'No action'),
  'min_train_classes': 12,
  'max_clips': None,
  'clips_per_class': 100,
  'sample_seed': 42,
  'train_epochs': 25,
  'backbone': 'vjepa',
  'backbones': ('vjepa', 'dinov2'),
  'embedding_mode': 'yolo_crop',
  'split_mode': 'subject',
  'use_class_weights': True,
  'include_human_labels': True,
  'min_confidence': 0.25,
  'skip_embeddings_if_exists': False,
  'skip_train_if_checkpoint_exists': False,
  'skip_eval_if_exists': False,
  'skip_if_cached': False,
  'eval_max_frames': 600,
  'infer_every': 16,
  'buffer_frames': 32,
  'dwell_windows': 2,
  'run_data_check': True,
  'run_embeddings': True,
  'run_train': True,
  'run_analysis': True,
  'run_compare': True,
  'analysis_split': 'random',
  'run_eval_video': False,
  'run_live_app': False,
  'live_webcam': None,
  'live_max_seconds': None,
  'checkpoint_name': 'har_vjepa_12c_crop_100each.pt',
  'eval_video_name': 

## Comparison table

In [4]:
import json
import pandas as pd
from IPython.display import display, Markdown

cmp_path = OUTPUTS_DIR / f"backbone_comparison_{RUN_TAG}.json"
if cmp_path.is_file():
    cmp = json.loads(cmp_path.read_text(encoding="utf-8"))
    display(pd.DataFrame(cmp["results"]))
    if "winner" in cmp:
        w = cmp["winner"]
        display(Markdown(f"**Winner:** `{w['backbone']}` — macro F1 **{w['macro_f1']:.4f}**, accuracy **{w['accuracy']:.4f}**"))
else:
    print("No comparison file — check pipeline status:", results.get("status"))

,backbone,checkpoint,status,split,n_test,accuracy,macro_f1,weighted_f1,emb_dim
0,vjepa,har_vjepa_12c_crop_100each.pt,ok,random,214,0.443925,0.409697,0.416703,1024
1,dinov2,har_dinov2_12c_crop_100each.pt,ok,random,215,0.595349,0.569990,0.572833,1024


**Winner:** `dinov2` — macro F1 **0.5700**, accuracy **0.5953**

In [5]:
from lib.har_analysis import compare_backbone_pair

display(compare_backbone_pair(clips_tag=RUN_TAG, split=ANALYSIS_SPLIT))

,backbone,checkpoint,status,split,n_test,accuracy,macro_f1,weighted_f1,emb_dim
0,vjepa,har_vjepa_12c_crop_100each.pt,ok,random,214,0.443925,0.409697,0.416703,1024
1,dinov2,har_dinov2_12c_crop_100each.pt,ok,random,215,0.595349,0.569990,0.572833,1024
